In [1]:
import networkx as nx
import re
from tqdm.notebook import tqdm
import pandas as pd

In [4]:
!wget -N https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz
!wget -N https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz.md5
!md5sum -c taxdump.tar.gz.md5

--2025-08-13 11:46:50--  https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.7, 130.14.250.10, 130.14.250.11, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 70409477 (67M) [application/x-gzip]
Saving to: ‘taxdump.tar.gz’

taxdump.tar.gz      100%[===================>]  67,15M  10,2MB/s    in 7,4s    

2025-08-13 11:46:58 (9,04 MB/s) - ‘taxdump.tar.gz’ saved [70409477/70409477]

--2025-08-13 11:46:59--  https://ftp.ncbi.nlm.nih.gov/pub/taxonomy/taxdump.tar.gz.md5
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.10, 130.14.250.11, 130.14.250.12, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.10|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 49 [application/x-gzip]
Saving to: ‘taxdump.tar.gz.md5’

taxdump.tar.gz.md5  100%[===================>]   

In [5]:
!tar xvzf taxdump.tar.gz
!rm citations.dmp delnodes.dmp division.dmp gencode.dmp images.dmp merged.dmp gc.prt readme.txt taxdump.tar.gz taxdump.tar.gz.md5

x citations.dmp
x delnodes.dmp
x division.dmp
x gencode.dmp
x images.dmp
x merged.dmp
x names.dmp
x nodes.dmp
x gc.prt
x readme.txt


In [4]:
taxonomy = nx.DiGraph()

with open('./nodes.dmp') as infile:
    for line in tqdm(infile):
        node1,node2,rank,*_ = re.split(r'\s\|\s', line)
        if node1 == node2:
            continue
        taxonomy.add_edge(node2, node1, rank=rank)

0it [00:00, ?it/s]

In [5]:
names = dict()
with open('./names.dmp') as infile:
    for line in tqdm(infile):
        parts = re.split(r'\s{0,}\|\s', line)
        taxid,name,_,nametype,_ = parts
        if 'scientific' in nametype:
            names[taxid] = name

0it [00:00, ?it/s]

In [6]:
def get_parents(node_id):
    yield node_id
    for parent in taxonomy.predecessors(node_id):
        yield from get_parents(parent)

with open('./taxonomy.tsv', 'w') as outfile:
    for node in tqdm(taxonomy.nodes):
        name = names[node]
        parents = list(get_parents(node))[::-1]
        outfile.write(f'{node}\t{name}\t{{{",".join(parents)}}}\n')


  0%|          | 0/2691268 [00:00<?, ?it/s]

In [7]:
nx.algorithms.is_arborescence(taxonomy)

True

In [18]:
df = pd.read_csv(
    '/Users/rensholmer/Documents/teaching/introduction_to_bioinformatics/sequence_generation/blast_results/Q9Z1T5.filtered_hits.tsv',
    sep='\t',
    index_col=0
)
all_taxids = {taxid for taxids in df.taxids for taxid in eval(taxids)}
all_parents = []
min_num_parents = 1e9
for taxid in all_taxids:
    parents = list(get_parents(taxid))[::-1]
    if '2787854' in parents: # other sequences such as synthetic constructs
        continue
    min_num_parents = min(min_num_parents, len(parents))
    all_parents.append(parents)
for pos in range(min_num_parents):
    if len({parents[pos] for parents in all_parents}) > 1:
        break
pos

NetworkXError: The node 346063 is not in the digraph.

In [ ]:
all_parents[0][pos - 1]

'9347'